# CURATOR - Phase B pilot on Kaggle

Run the cells **in order**. Settings (right sidebar): **Accelerator = GPU T4 x2**, **Internet = On**,
and attach the Dataset that contains `curator-rl-bundle.zip` (see `docs/KAGGLE.md`).

What this measures: base-model pass rates and the cost per prompt of every environment
(decides the model, the round horizon, and the real costs that replace the simulator's
placeholders), plus a first contact with TRL's GRPOTrainer. It does **not** train anything
that matters and never touches the sealed test set.

In [ ]:
# 1) Hardware and library check
!nvidia-smi
import sys, torch, transformers
print(sys.version.split()[0], "| torch", torch.__version__, "| transformers", transformers.__version__)
print("cuda:", torch.cuda.is_available(), "| gpus:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# 2) Unpack the bundle into the working directory and self-check
import os, shutil, glob, zipfile, json

WORK = "/kaggle/working/curator-rl"
if os.path.exists(WORK):
    shutil.rmtree(WORK)
zips = glob.glob("/kaggle/input/**/curator-rl-bundle.zip", recursive=True)
if zips:                                    # Dataset kept the zip as-is
    os.makedirs(WORK)
    zipfile.ZipFile(zips[0]).extractall(WORK)
else:                                       # Kaggle already extracted the zip into the dataset folder
    roots = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)]
    assert roots, "bundle not found: attach the Dataset that holds curator-rl-bundle.zip"
    shutil.copytree(roots[0], WORK)
print(open(WORK + "/BUNDLE_INFO.json").read())
assert not os.path.exists(WORK + "/data/test_sealed"), "sealed test set must never be on Kaggle"

!pip install -q pydantic pyyaml
%cd /kaggle/working/curator-rl
!python -c "import sys; sys.path.insert(0,'src'); from curator_rl.core.config import load_config; c=load_config('configs/base.yaml'); print('config ok, gamma', c.scheduler.gamma)"

## 3) The pilot
Two models are measured **in parallel, one per GPU**: Qwen2.5-0.5B-Instruct (the plan) and
Qwen2.5-1.5B-Instruct (the fallback if 0.5B turns out to be too weak). Expect roughly 10-25 minutes.
If your session shows only one GPU, run the two commands one after the other instead.

In [ ]:
%%bash
cd /kaggle/working/curator-rl
mkdir -p /kaggle/working/pilot
CUDA_VISIBLE_DEVICES=0 python scripts/kaggle_pilot.py --model Qwen/Qwen2.5-0.5B-Instruct \
    --envs gsm8k,math35,mbpp,countdown,noisy --n-prompts 64 --group-size 8 --max-new-tokens 512 \
    --out /kaggle/working/pilot > /kaggle/working/pilot/log_0p5b.txt 2>&1 &
CUDA_VISIBLE_DEVICES=1 python scripts/kaggle_pilot.py --model Qwen/Qwen2.5-1.5B-Instruct \
    --envs gsm8k,math35,mbpp,countdown,noisy --n-prompts 64 --group-size 8 --max-new-tokens 512 \
    --out /kaggle/working/pilot > /kaggle/working/pilot/log_1p5b.txt 2>&1 &
wait
echo "---- 0.5B ----"; tail -n 14 /kaggle/working/pilot/log_0p5b.txt
echo "---- 1.5B ----"; tail -n 14 /kaggle/working/pilot/log_1p5b.txt

In [ ]:
# 4) Show the result tables (copy these back to the chat)
from pathlib import Path
for f in sorted(Path("/kaggle/working/pilot").glob("pilot_*.md")):
    print(f.read_text())

## 5) TRL spike (first contact with GRPOTrainer)
Installs TRL, then runs a 3-step LoRA GRPO smoke test on GPU 0. A failure here is useful data:
the probe records the error text and traceback instead of stopping. It has never been run against
TRL before, so expect to iterate once.

In [ ]:
!pip uninstall -y -q torchao
!pip install -q trl
%cd /kaggle/working/curator-rl
!CUDA_VISIBLE_DEVICES=0 python scripts/kaggle_trl_probe.py --out /kaggle/working/trl_probe

In [ ]:
# 6) Collect everything to send back
import json, glob
print(open("/kaggle/working/trl_probe/trl_probe.json").read()[:6000])
print(sorted(glob.glob("/kaggle/working/pilot/*") + glob.glob("/kaggle/working/trl_probe/*")))